# Session 1: Introduction to MongoDB

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-01_first_look.py`

In [2]:
# Session 1 · A first look at documents, collections and databases
import mongomock
from store_db import show

client = mongomock.MongoClient()          # real server: pymongo.MongoClient("mongodb://localhost:27017")
db = client["kgs"]                        # mongosh: use kgs
print("Databases before any insert:", client.list_database_names())

# A document is a JSON-like object. Fields can hold arrays and sub-documents.
doc = {
    "name": "Wireless Mouse",
    "category": "Accessories",
    "price": 1500,
    "tags": ["wireless", "office"],          # array
    "specs": {"brand": "Logi", "dpi": 1600}, # embedded document
}
result = db.products.insert_one(doc)      # mongosh: db.products.insertOne({...})
print("Inserted _id:", result.inserted_id, "| type:", type(result.inserted_id).__name__)

# Flexible schema: the next document has different fields and that is allowed.
db.products.insert_one({"name": "Gift Card", "category": "Vouchers", "value": 5000, "expires": "2026-12-31"})

print("Databases after insert:", client.list_database_names())
print("Collections in kgs:", db.list_collection_names())
print("Documents in products:", db.products.count_documents({}))

print("\nAll products (without _id):")
show(db.products.find({}, {"_id": 0}))   # mongosh: db.products.find({}, {_id: 0})

# Read the embedded field with dot notation
print("\nBrand of the mouse:", db.products.find_one({"name": "Wireless Mouse"})["specs"]["brand"])

Databases before any insert: []
Inserted _id: 6adcbf8db5747a37009cfec7 | type: ObjectId
Databases after insert: ['kgs']
Collections in kgs: ['products']
Documents in products: 2

All products (without _id):
{"name": "Wireless Mouse", "category": "Accessories", "price": 1500, "tags": ["wireless", "office"], "specs": {"brand": "Logi", "dpi": 1600}}
{"name": "Gift Card", "category": "Vouchers", "value": 5000, "expires": "2026-12-31"}
(2 documents)

Brand of the mouse: Logi


## Example: `session-01_relational_vs_document.py`

In [3]:
# Session 1 · The same order as relational rows and as one MongoDB document
import json

# Relational design: 3 tables, joined by keys
orders      = [{"OrderID": 5001, "CustomerID": 1, "OrderDate": "2025-03-02"}]
order_items = [{"OrderID": 5001, "ProductID": 101, "Qty": 2, "Price": 1500},
               {"OrderID": 5001, "ProductID": 107, "Qty": 1, "Price": 4500}]
customers   = [{"CustomerID": 1, "Name": "Ayesha Khan", "City": "Karachi"}]
print("Relational: rows spread over", 3, "tables ->",
      len(orders) + len(order_items) + len(customers), "rows to read with 2 joins")

# Document design: everything one screen needs lives in one document
cust = customers[0]
document = {
    "_id": orders[0]["OrderID"],
    "orderDate": orders[0]["OrderDate"],
    "customer": {"id": cust["CustomerID"], "name": cust["Name"], "city": cust["City"]},  # embedded copy
    "items": [{"productId": r["ProductID"], "qty": r["Qty"], "price": r["Price"]} for r in order_items],
}
document["total"] = sum(i["qty"] * i["price"] for i in document["items"])
print("Document: 1 read, no joins\n")
print(json.dumps(document, indent=2))

# Terminology map
pairs = [("Database", "Database"), ("Table", "Collection"), ("Row", "Document"),
         ("Column", "Field"), ("Primary key", "_id"), ("JOIN", "Embedding or $lookup"),
         ("Index", "Index")]
print("\n{:<12} | {}".format("RDBMS", "MongoDB"))
print("-" * 13 + "+" + "-" * 22)
for a, b in pairs:
    print("{:<12} | {}".format(a, b))

Relational: rows spread over 3 tables -> 4 rows to read with 2 joins
Document: 1 read, no joins

{
  "_id": 5001,
  "orderDate": "2025-03-02",
  "customer": {
    "id": 1,
    "name": "Ayesha Khan",
    "city": "Karachi"
  },
  "items": [
    {
      "productId": 101,
      "qty": 2,
      "price": 1500
    },
    {
      "productId": 107,
      "qty": 1,
      "price": 4500
    }
  ],
  "total": 7500
}

RDBMS        | MongoDB
-------------+----------------------
Database     | Database
Table        | Collection
Row          | Document
Column       | Field
Primary key  | _id
JOIN         | Embedding or $lookup
Index        | Index
